# TokenGate: find a needle, read fewer tokens

[![GitHub](https://img.shields.io/badge/GitHub-TokenGate-181717?logo=github&logoColor=white)](https://github.com/token-gate-llms/TokenGate) [![arXiv](https://img.shields.io/badge/arXiv-2610.26754-B31B1B?logo=arxiv&logoColor=white)](https://arxiv.org/abs/2610.26754) [![X](https://img.shields.io/badge/X-Launch%20thread-000000?logo=x&logoColor=white)](https://x.com/lianapatel_/status/2112924355240099875) [![HF Qwen3](https://img.shields.io/badge/%F0%9F%A4%97-Qwen3--1.7B-FFD21E)](https://huggingface.co/liana313/tokengate-niah-qwen3-1.7b) [![HF Qwen3.5](https://img.shields.io/badge/%F0%9F%A4%97-Qwen3.5--2B-FFD21E)](https://huggingface.co/liana313/tokengate-niah-qwen3.5-2b)

TokenGate adds a learned gate near the start of an existing model. It scores the document, keeps the most useful **K tokens**, and lets later layers process those tokens. The question and instructions stay intact. This demo uses the released NIAH gate and its paired LoRA adapter.

![TokenGate overview](https://raw.githubusercontent.com/token-gate-llms/TokenGate/main/assets/fig1_overview.png)

Run the cells in order. Sections: **Setup and information → Inference → Optional training**. The default targets a free Colab T4; training needs an A100-class GPU.

## 1. Setup and information

**NIAH task:** one sentence is planted in a long document made from held-out essays. The model answers a question about that sentence. We use **three different city facts**, with the needle near the beginning, middle, and end of separate documents. At 16k, all three rows come from the frozen 150-row test partition. K is the maximum number of **document** tokens TokenGate keeps for later layers; `None` keeps all.

Qwen3.5-2B has a [released NIAH gate and paired LoRA](https://huggingface.co/liana313/tokengate-niah-qwen3.5-2b). It requires the A100 option. Qwen3-1.7B is the tested free-T4 default.

In [ ]:
#@title Demo settings { display-mode: "form" }
MODEL = 'Qwen3-1.7B' #@param ['Qwen3-1.7B', 'Qwen3.5-2B']
DOC_TOKENS = '16k' #@param ['2k', '4k', '8k', '16k']
KEEP_BUDGETS = '100, 500, 2000' #@param {type:"string"}
GPU_RUNTIME = 'T4' #@param ['T4', 'A100']

BASE_MODELS = {'Qwen3-1.7B':'Qwen/Qwen3-1.7B', 'Qwen3.5-2B':'Qwen/Qwen3.5-2B'}
CHECKPOINTS = {'Qwen3-1.7B':('liana313/tokengate-niah-qwen3-1.7b','3ef4d84ebd161a9700acced6e92f7cd5c50225ad'),
               'Qwen3.5-2B':('liana313/tokengate-niah-qwen3.5-2b','b1fe3e72b9896efc821fbfa02a19fcded4de6d77')}
BASE_MODEL = BASE_MODELS[MODEL]
assert MODEL != 'Qwen3.5-2B' or GPU_RUNTIME == 'A100', 'Qwen3.5 requires the A100 runtime.'
DOC_TOKENS = int(DOC_TOKENS[:-1]) * 1024
BUDGETS = tuple(int(x.strip()) for x in KEEP_BUDGETS.split(',') if x.strip())
assert BUDGETS and len(set(BUDGETS)) == len(BUDGETS) and all(k > 0 for k in BUDGETS)
print(BASE_MODEL, '|', DOC_TOKENS, 'document tokens | K =', BUDGETS, '| GPU:', GPU_RUNTIME)

In [ ]:
#@title Install and import the API { display-mode: "form" }
import gc, subprocess, sys
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Choose a GPU runtime first.'
assert GPU_RUNTIME in torch.cuda.get_device_name(0), f'Select a {GPU_RUNTIME} GPU in Colab: Runtime → Change runtime type.'
SOURCE = Path.cwd() if (Path.cwd()/'tokengate/demo.py').exists() else Path('/content/TokenGate')
if SOURCE == Path('/content/TokenGate'):
    if not SOURCE.exists():
        subprocess.run(['git','clone','-q','--branch','codex/colab-api-demo',
                        'https://github.com/token-gate-llms/TokenGate',str(SOURCE)],check=True)
    else:
        subprocess.run(['git','-C',str(SOURCE),'pull','--ff-only'],check=True)
assert (SOURCE/'tokengate/demo.py').exists(), 'TokenGate demo helper is missing from this checkout.'
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(SOURCE),
                'transformers==5.5.0','matplotlib'],check=True)
sys.path.insert(0,str(SOURCE))
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import snapshot_download
from tokengate import load_model, TokenGateConfig, TokenGateTrainer
from tokengate.demo import (niah_examples, niah_prompt, plain_results, gate_results,
                            result_figure, show_kept_tokens, training_rows)
print('GPU:', torch.cuda.get_device_name(0))

## 2. Inference

### Ordinary model: read the whole document

The unmodified base model answers the same three NIAH questions first. Its time to first token is the reference for the TokenGate results below.

In [ ]:
#@title Make three held-out NIAH prompts { display-mode: "form" }
samples = niah_examples(SOURCE, DOC_TOKENS)
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
pairs = [niah_prompt(tok, row) for row in samples]
gate_prompts, plain_prompts = map(list, zip(*pairs))
for row in samples:
    print(f"{row['city']:7s} needle at {row['depth_actual']:.0%} · {row['question']}")

In [ ]:
#@title Run the ordinary model { display-mode: "form" }
if GPU_RUNTIME == 'A100':
    assert torch.cuda.get_device_properties(0).total_memory >= 35*2**30, 'Use an A100-class GPU for single-pass prefill.'
dtype = torch.float16 if GPU_RUNTIME == 'T4' else torch.bfloat16
plain_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=dtype,
    attn_implementation='sdpa').cuda().eval()
plain = plain_results(plain_model, tok, plain_prompts, samples,
    chunk_tokens=256 if GPU_RUNTIME == 'T4' else None)
for row in plain: print(f"{row['city']:7s} correct={bool(row['correct'])}  TTFT={row['prefill_s']:.2f}s")
del plain_model; gc.collect(); torch.cuda.empty_cache()

### Accelerate with TokenGate

`load_model` loads the base model, gate, and paired LoRA. `generate` uses the same prompts; `max_kept_prefill_tokens=K` controls how many document tokens survive prefill. The all-token arm shows the effect of the paired adapter without pruning.

In [ ]:
#@title Run TokenGate at each token budget { display-mode: "form" }
repo_id, revision = CHECKPOINTS[MODEL]
checkpoint = snapshot_download(repo_id, revision=revision)
gate = load_model(checkpoint, gate_attn='sdpa' if GPU_RUNTIME == 'T4' else 'flex', dtype=dtype)
gated = gate_results(gate, tok, gate_prompts, samples, BUDGETS)
for row in gated:
    print(f"{row['city']:7s} K={str(row['budget']):5s} correct={bool(row['correct'])} "
          f"TTFT={row['prefill_s']:.2f}s retained={row['retained']:.1%}")

### Interpret the results

The bars compare **time to first token** on the same three prompts and GPU; labels also show retrieval accuracy. On T4, the ordinary model reads the full document in cached 256-token chunks because single-pass prefill ran out of memory at the default 16k length. The paper uses a single-pass baseline, so **T4 speedups are not directly comparable to the paper's**. On A100, the ordinary model uses single-pass prefill. Three examples show the mechanism, not a reliable accuracy estimate.

In [ ]:
#@title Accuracy and speedup { display-mode: "form" }
from IPython.display import display
display(result_figure(plain, gated,
    baseline_name='chunked base (T4)' if GPU_RUNTIME == 'T4' else 'single-pass base (A100)'))

In [ ]:
#@title Which document tokens did the gate keep? { display-mode: "form" }
show_kept_tokens(gate, tok, gate_prompts[0], samples[0], budget=min(BUDGETS))

## 3. Optional training

Training uses the public `TokenGateConfig` and `TokenGateTrainer` API on new synthetic NIAH rows. It needs an A100-class GPU. The 128-example, one-epoch default is a **workflow check**, not the released 12,000-example recipe; it may not learn reliable retrieval. For the full recipe, use [the A100 training notebook](https://github.com/token-gate-llms/TokenGate/blob/main/notebooks/train_niah_release_recipe.ipynb).

In [ ]:
#@title Train your own gate (A100 only) { display-mode: "form" }
RUN_TRAINING = False #@param {type:"boolean"}
TRAIN_EXAMPLES = 128 #@param {type:"integer"}
if RUN_TRAINING:
    assert torch.cuda.get_device_properties(0).total_memory >= 35*2**30, 'Training needs an A100-class GPU.'
    del gate; gc.collect(); torch.cuda.empty_cache()
    work = Path('/content/tokengate-training') if Path('/content').is_dir() else Path.cwd()/'tokengate-training'
    train_data, val_data = training_rows(SOURCE, work, n_train=TRAIN_EXAMPLES)
    config = TokenGateConfig(base_model=BASE_MODEL, gate_layer=1,
        output_dir=str(work/'checkpoint'), learning_rate=1e-4, epochs=1,
        k_eval=[500], lora_rank=16, lora_alpha=32, lora_lr=1e-4,
        lora_targets=['q_proj','v_proj'], num_gpus=1,
        batch_size=1, grad_accum=1, grad_checkpoint=True,
        warmup_steps=4, eval_freq=TRAIN_EXAMPLES,
        checkpoint_freq=TRAIN_EXAMPLES, eval_gen_batch_size=1,
        eval_val_samples=len(val_data), max_gen_tokens=64, max_answer_tokens=64,
        data_root=str(work/'data'), attn_backend='flex',
        compile_phase2=False, pack_phase2=False, wandb_project=None)
    trainer = TokenGateTrainer(config, train_dataset=train_data,
        eval_dataset=val_data, task='niah_kamradt')
    trained_checkpoint = trainer.train()/'gate_final.pt'
    print('Saved:', trained_checkpoint)
else:
    print('Training skipped; inference above used the released checkpoint.')

In [ ]:
#@title Try your trained gate { display-mode: "form" }
if RUN_TRAINING:
    own_gate = load_model(str(trained_checkpoint), gate_attn='flex', dtype=torch.bfloat16)
    own = gate_results(own_gate, tok, gate_prompts, samples, BUDGETS)
    for row in own:
        print(f"{row['city']:7s} K={str(row['budget']):5s} correct={bool(row['correct'])} "
              f"TTFT={row['prefill_s']:.2f}s")
else:
    print('Run the training cell first to test a new gate.')